# Phase 6 — ATM Simulation
**Goal:** Generate realistic ATM fraud scenarios as the 3rd dataset and run them through the agent.

In [ ]:
import pandas as pd
import numpy as np
import joblib, os, warnings
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

agent = joblib.load('../models/agentic_system.pkl')
xgb_ulb  = agent['ulb_model']
xgb_ieee = agent['ieee_model']
print("Agentic system loaded!")

In [ ]:
# Generate simulated ATM transactions
np.random.seed(42)
n_normal = 950
n_fraud  = 50
n_total  = n_normal + n_fraud

def generate_transactions(n, is_fraud=False):
    rows = []
    base_time = datetime(2024, 1, 1, 9, 0, 0)
    for i in range(n):
        if is_fraud:
            # Fraud patterns: late night, large amounts, rapid transactions
            hour     = np.random.choice([0,1,2,3,22,23], p=[0.2,0.2,0.2,0.15,0.15,0.1])
            amount   = np.random.choice([
                np.random.uniform(15000, 25000),  # large withdrawal
                np.random.uniform(100, 500),       # rapid small withdrawals
            ], p=[0.6, 0.4])
            location_change = np.random.choice([1, 0], p=[0.8, 0.2])
            rapid_txn       = np.random.choice([1, 0], p=[0.7, 0.3])
            failed_attempts = np.random.randint(1, 5)
            v_features      = np.random.normal(-2, 2, 28)  # abnormal PCA features
        else:
            hour     = np.random.randint(8, 22)
            amount   = np.random.exponential(500)
            amount   = min(amount, 10000)
            location_change = np.random.choice([1, 0], p=[0.05, 0.95])
            rapid_txn       = np.random.choice([1, 0], p=[0.05, 0.95])
            failed_attempts = 0
            v_features      = np.random.normal(0, 1, 28)

        row = {
            'transaction_id' : f"ATM_{'F' if is_fraud else 'N'}_{i+1:04d}",
            'timestamp'      : (base_time + timedelta(hours=i*0.5)).strftime('%Y-%m-%d %H:%M:%S'),
            'amount'         : round(abs(amount), 2),
            'hour_of_day'    : hour,
            'location_change': location_change,
            'rapid_txn_flag' : rapid_txn,
            'failed_attempts': failed_attempts,
            'is_fraud'       : 1 if is_fraud else 0,
        }
        for j, v in enumerate(v_features, 1):
            row[f'V{j}'] = round(v, 6)
        rows.append(row)
    return rows

normal_txns = generate_transactions(n_normal, is_fraud=False)
fraud_txns  = generate_transactions(n_fraud,  is_fraud=True)
df_sim = pd.DataFrame(normal_txns + fraud_txns).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Generated {len(df_sim)} ATM transactions")
print(f"Fraud: {df_sim['is_fraud'].sum()} | Normal: {(df_sim['is_fraud']==0).sum()}")
df_sim.to_csv('../data/simulated_atm.csv', index=False)
print("Simulated dataset saved to data/simulated_atm.csv")

In [ ]:
# Visualize simulated data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Amount distribution
fraud_amt  = df_sim[df_sim['is_fraud']==1]['amount']
normal_amt = df_sim[df_sim['is_fraud']==0]['amount']
axes[0].hist(normal_amt, bins=50, alpha=0.7, color='steelblue', label='Normal')
axes[0].hist(fraud_amt,  bins=50, alpha=0.7, color='crimson',   label='Fraud')
axes[0].set_title('Amount Distribution'); axes[0].legend()

# Hour distribution
axes[1].hist(df_sim[df_sim['is_fraud']==0]['hour_of_day'], bins=24, alpha=0.7, color='steelblue', label='Normal')
axes[1].hist(df_sim[df_sim['is_fraud']==1]['hour_of_day'], bins=24, alpha=0.7, color='crimson',   label='Fraud')
axes[1].set_title('Transaction Hour'); axes[1].legend()

# Fraud vs normal count
counts = df_sim['is_fraud'].value_counts()
axes[2].bar(['Normal','Fraud'], counts.values, color=['steelblue','crimson'], edgecolor='black')
axes[2].set_title('Fraud vs Normal Count')

plt.tight_layout()
plt.savefig('../reports/phase6_simulated_data.png', dpi=150)
plt.show()
print("Simulation chart saved!")

In [ ]:
# Run simulated transactions through Agentic AI
# Prepare features — use V1-V28 padded to match both models
ulb_n  = xgb_ulb.n_features_in_
ieee_n = xgb_ieee.n_features_in_

def pad_features(f, n):
    if len(f) < n: return np.concatenate([f, np.zeros(n-len(f))])
    return f[:n]

LEDGER_PATH = '../data/fraud_ledger.csv'

def log_sim(txn_id, amount, hour, ulb_p, ieee_p, final_p, action):
    pd.DataFrame([{
        'timestamp':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'transaction_id':txn_id, 'amount':amount, 'hour':hour,
        'dataset_source':'Simulated-ATM',
        'ulb_fraud_prob':round(ulb_p,4), 'ieee_fraud_prob':round(ieee_p,4),
        'final_fraud_prob':round(final_p,4),
        'action':action, 'reason':f'Simulated ATM transaction — prob={final_p:.4f}'
    }]).to_csv(LEDGER_PATH, mode='a', header=False, index=False)

v_cols = [f'V{i}' for i in range(1,29)]
results_sim = []
print("Running 1000 simulated transactions through Agentic AI...")
for _, row in df_sim.iterrows():
    v_features = row[v_cols].values.astype(float)
    f_ulb      = pad_features(v_features, ulb_n)
    f_ieee     = pad_features(v_features, ieee_n)

    ulb_p   = float(xgb_ulb.predict_proba(f_ulb.reshape(1,-1))[0][1])
    ieee_p  = float(xgb_ieee.predict_proba(f_ieee.reshape(1,-1))[0][1])
    final_p = 0.5*ulb_p + 0.5*ieee_p

    if   final_p > 0.8: action = 'BLOCK'
    elif final_p > 0.5: action = 'OTP_CHALLENGE'
    else:               action = 'ALLOW'

    log_sim(row['transaction_id'], row['amount'], row['hour_of_day'], ulb_p, ieee_p, final_p, action)
    results_sim.append({'id':row['transaction_id'], 'actual':row['is_fraud'],
                        'final_prob':round(final_p,4), 'action':action})

df_results = pd.DataFrame(results_sim)
print(f"Done! Processed {len(df_results)} transactions")
print("\nAction breakdown:")
print(df_results['action'].value_counts())

In [ ]:
# Evaluate agent on simulated data
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Map actions to predictions (BLOCK/OTP = predicted fraud, ALLOW = predicted normal)
df_results['predicted'] = df_results['action'].map(
    {'BLOCK':1, 'OTP_CHALLENGE':1, 'ALLOW':0})

print("=== Agent Performance on Simulated ATM Data ===")
print(classification_report(df_results['actual'], df_results['predicted'],
                             target_names=['Normal','Fraud']))

cm = confusion_matrix(df_results['actual'], df_results['predicted'])
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Fraud'], yticklabels=['Normal','Fraud'])
plt.title('Confusion Matrix — Simulated ATM')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../reports/phase6_simulation_confusion.png', dpi=150)
plt.show()
print("Phase 6 complete!")